1- reading the dataset form (.csv) file

In [ ]:
import pandas as pd
data=pd.read_csv('filename.csv')

2- Split the features and the target in variables (X_test) and (y_test)

In [ ]:
X_test = data.drop("charges",axis=1)
y_test = data['charges']

3- Load the parameters, transformers, and functions used to clean the training and validation data, to use them to preprocess the test data

In [ ]:
pip install dill

In [ ]:
# Library needed to load this type of file
import pickle
import dill

# Load the saved preprocessing objects
with open('scaler.pkl', 'rb') as file:
    transformer_scale = pickle.load(file)

with open('numerical_imputer.pkl', 'rb') as file:
    numerical_imputer = pickle.load(file)

with open('categorical_imputer.pkl', 'rb') as file:
    categorical_imputer = pickle.load(file)

with open('label_encoder_sex.pkl', 'rb') as file:
    transformer_label_encoder_sex = pickle.load(file)

with open('label_encoder_smoker.pkl', 'rb') as file:
    transformer_label_encoder_smoker = pickle.load(file)

with open('one_hot_encoder_region.pkl', 'rb') as file:
    transformer_one_hot_encoder_region = pickle.load(file)

# Load the function from the file
with open('remove_outliers.pkl', 'rb') as file:
    remove_outliers = dill.load(file)

with open('clean_categorical_typos.pkl', 'rb') as file:
    clean_categorical_typos = dill.load(file)
    
with open('binary_encoding.pkl', 'rb') as file:
    binary_encoding = dill.load(file)

with open('one_hot_encoding.pkl', 'rb') as file:
    one_hot_encoding = dill.load(file)

# Load the medians from a pickle file
with open("medians.pkl", "rb") as file:
    medians = pickle.load(file)

4- Apply the saved transformers to the test dataset to clean it

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder

def preprocessing_test(test_data, y_test, transformer_label_encoder_sex, transformer_label_encoder_smoker,
                       transformer_one_hot_encoder_region, numerical_imputer, categorical_imputer, transformer_scale, medians):
    test_data_cp = test_data.copy()

    # Convert y_test to a DataFrame to handle outliers (same as validation)
    y_test_df = pd.DataFrame(y_test, columns=["charges"])
    
    numerical_cols = ['age', 'bmi', 'children']
    categorical_cols = ['sex', 'smoker', 'region']

    # Imputation using transformers already fitted during training (do not re-fit)
    test_data_cp[numerical_cols] = numerical_imputer.transform(test_data_cp[numerical_cols])
    test_data_cp[categorical_cols] = categorical_imputer.transform(test_data_cp[categorical_cols])

    
    # Clean outliers for testing data and replace it with the median of the column
    for col in numerical_cols:
        test_data_cp = remove_outliers(test_data_cp, col, median_value=medians[col])


    # Clean categorical typos for testing data
    for col in categorical_cols:
        test_data_cp = clean_categorical_typos(test_data_cp, col)

    # Clean outliers in y_test (charges), if necessary
    y_test_df = remove_outliers(y_test_df, "charges", median_value=medians["charges"])

    # Scaling using the already trained scaler (no recalculation)
    test_data_cp = transformer_scale.transform(test_data_cp)
    test_data_cp = pd.DataFrame(test_data_cp, columns=numerical_cols + categorical_cols)

    # Encoding using the trained transformers (do not re-fit)
    test_data_cp = binary_encoding(test_data_cp, 'sex', transformer_label_encoder_sex)
    test_data_cp = binary_encoding(test_data_cp, 'smoker', transformer_label_encoder_smoker)
    test_data_cp = one_hot_encoding(test_data_cp, 'region', transformer_one_hot_encoder_region)

    # Convert y_test_df back to Series for consistency in return type
    y_test_processed = y_test_df["charges"]

    return test_data_cp, y_test_processed


In [ ]:
X_test_transformed, y_test_transformed = preprocessing_test(X_test, y_test, transformer_label_encoder_sex, transformer_label_encoder_smoker, transformer_one_hot_encoder_region, numerical_imputer, categorical_imputer, transformer_scale, medians)

5- Load the trained model

In [ ]:
from tensorflow import keras

loaded_model = keras.models.load_model('trained_model.keras')
# Display a summary of the model
loaded_model.summary()

6- Use the trained model to predict charges in the test data

In [ ]:
predictions = loaded_model.predict(X_test_transformed)

7- Evaluate and print the mean absolute error.

In [ ]:
test_loss, test_mae = loaded_model.evaluate(X_test_transformed, y_test_transformed)

print(f'Mean Absolute Error (MAE) on the test set: {test_mae:.4f}')

8- Print the real charges and the predicted charges.

In [ ]:
for actual, pred in zip (y_test_transformed, predictions):
    print(f'Actual: {actual}, pred: {pred}') 

9- Display a scatter plot to compare between predicted and actual charges.

In [ ]:
import matplotlib.pyplot as plt

# Create a scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test_transformed, predictions, color='blue', alpha=0.7)

# Add a line for perfect predictions (where predicted == actual)
plt.plot([min(y_test_transformed), max(y_test_transformed)], [min(y_test_transformed), max(y_test_transformed)], color='red', linestyle='--')

# Add labels and title
plt.xlabel('Actual Charges Values')
plt.ylabel('Predicted Charges Values')
plt.title('Predicted vs Actual Charges Values')

# Show the plot
plt.show()